In [8]:
print(sys.executable)

/Users/sskim/anaconda3/envs/motornet/bin/python


In [7]:
import os
import sys
import json
import numpy as np
import torch as th
import matplotlib.pyplot as plt
import motornet as mn
from utils import run_rollout, plot_simulations
from importlib import reload
import agents
import utils
reload(utils)
reload(agents)
from agents import SLAgent
from tasks import CentreOutFF

# from sklearn.decomposition import PCA

In [ ]:
# --- device setting and config loading ---
device = th.device("cuda" if th.cuda.is_available() else "cpu")
config = 'params.json'
print(f"Using device: {device}")
with open(config, 'r') as f:
    config = json.load(f)

n_obs, n_actions = 17, 6

# --- parameters setting ---
env_params = config['env_params']
train_params = config['training_params']
effector = mn.effector.RigidTendonArm26(muscle=mn.muscle.RigidTendonHillMuscle())
env = CentreOutFF(effector=effector, **env_params)
agentSL = SLAgent(n_obs, n_actions)

agentSL.load('agent_baseline_18000.pth')
fig, axes = plt.subplots(1, 2, figsize=(10,3))
data = run_rollout(env, agentSL, batch_size = 64, condition='test',ff_coefficient = 0., catch_trial_perc=0.0)
plt.figure(figsize=(10,3))
plot_simulations(data, axes[0])
data = run_rollout(env, agentSL, batch_size = 64, condition='test',ff_coefficient = 8., catch_trial_perc=0.0)
plot_simulations(data, axes[1])



Using device: cpu


In [9]:
def analyze_hidden_states(hidden_states_data):
    """
    주어진 hidden state 데이터의 주성분 궤적을 분석하고 시각화합니다.

    Args:
        hidden_states_data (np.ndarray): 분석할 은닉 상태 데이터. 
                                         형태: (batch_size, time_steps, hidden_dims)
    """
    print("은닉 상태 분석 시작...")
    
    # 1. 데이터 형태 확인
    if hidden_states_data.ndim != 3:
        raise ValueError("입력 데이터는 반드시 (batch_size, time_steps, hidden_dims) 형태의 3차원 배열이어야 합니다.")
    
    batch_size, time_steps, hidden_dims = hidden_states_data.shape
    print(f"데이터 형태 확인: Batch={batch_size}, Time Steps={time_steps}, Hidden Dims={hidden_dims}")

    # 2. 데이터 처리 및 PCA
    # PCA를 위해 데이터를 (batch * steps, hidden_dims) 형태로 변환해야 합니다.
    # 올바른 궤적을 위해 (batch, steps, dims) -> (steps, batch, dims)로 축을 바꾼 뒤 reshape합니다.
    reshaped_states = hidden_states_data.transpose(1, 0, 2).reshape(-1, hidden_dims)
    
    print(f"PCA 수행 대상 데이터 형태: {reshaped_states.shape}")
    pca = PCA(n_components=2) # 상위 2개의 주성분 추출
    hidden_states_2d = pca.fit_transform(reshaped_states)
    
    print(f"설명된 분산 비율 (PC1, PC2): {pca.explained_variance_ratio_}")

    # 3. 시각화
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # 각 배치의 궤적을 따로 그림
    for i in range(batch_size):
        # i번째 배치의 궤적 데이터 추출
        # Reshape된 데이터에서 i번째 배치의 데이터를 올바르게 가져옵니다.
        trajectory = hidden_states_2d[i::batch_size]
        
        # 시간에 따라 색상이 변하도록 컬러맵 설정 (파랑 -> 빨강)
        colors = plt.cm.viridis(np.linspace(0, 1, len(trajectory)))
        
        # 선분 하나씩 그리면서 색상 적용
        for j in range(len(trajectory) - 1):
            ax.plot(trajectory[j:j+2, 0], trajectory[j:j+2, 1], color=colors[j], alpha=0.6, linewidth=1.5)

    # 시작점과 끝점 표시
    start_points = hidden_states_2d[0:batch_size]
    end_points = hidden_states_2d[-batch_size:]
    ax.scatter(start_points[:, 0], start_points[:, 1], c='blue', s=100, label='Start (t=0)', zorder=3, ec='w')
    ax.scatter(end_points[:, 0], end_points[:, 1], c='red', s=100, label=f'End (t={time_steps-1})', zorder=3, ec='w')
    
    ax.set_title("GRU Hidden State Trajectory (PCA)", fontsize=16)
    ax.set_xlabel("Principal Component 1", fontsize=12)
    ax.set_ylabel("Principal Component 2", fontsize=12)
    ax.legend()
    ax.grid(True)
    plt.show()

In [ ]:
hidden_states_data = data['all_hidden'].detach().cpu().numpy()
batch_size, time_steps, hidden_dims = hidden_states_data.shape

reshaped_states = hidden_states_data.transpose(1, 0, 2).reshape(-1, hidden_dims)
    
print(f"PCA 수행 대상 데이터 형태: {reshaped_states.shape}")
pca = PCA(n_components=2) # 상위 2개의 주성분 추출
hidden_states_2d = pca.fit_transform(reshaped_states)

print(f"설명된 분산 비율 (PC1, PC2): {pca.explained_variance_ratio_}")

# 3. 시각화
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(10, 8))

for i in range(batch_size):
# for i in range(4):
        # i번째 배치의 궤적 데이터 추출
        # Reshape된 데이터에서 i번째 배치의 데이터를 올바르게 가져옵니다.
        trajectory = hidden_states_2d[i::batch_size]
        
        # 시간에 따라 색상이 변하도록 컬러맵 설정 (파랑 -> 빨강)
        colors = plt.cm.viridis(np.linspace(0, 1, len(trajectory)))
        
        # 선분 하나씩 그리면서 색상 적용
        for j in range(len(trajectory) - 1):
            ax.plot(trajectory[j:j+2, 0], trajectory[j:j+2, 1], color=colors[j], alpha=0.6, linewidth=1.5)
            # ax.plot(trajectory[:, 0], trajectory[:, 1], color=colors[j], alpha=0.6, linewidth=1.5)